# I. YANN: Budowanie i trenowanie sieci
YANN implementuje wielowarstwowy perceptron (MLP), czyli sieć neuronową złożoną z gęstych, w pełni połączonych warstw.
## Struktura sieci
Strukturę sieci opisuje lista rozmiarów warstw:
`network_structure=[2, 4, 1]`

Oznacza ona 2 cechy wejściowe, 4 neurony w warstwie ukrytej i 1 wyjście. Pierwsza liczba określa jedynie rozmiar wejścia - dla warstwy wejściowej nie podajemy funkcji aktywacji. Każda kolejna warstwa ma własną macierz wag, wektor biasów i funkcję aktywacji. Każdy neuron otrzymuje sygnał od wszystkich neuronów poprzedniej warstwy, a wyjście jednej warstwy staje się wejściem następnej. Warstw ukrytych może być dowolnie wiele.
Sieć przechowuje swoje parametry oraz wartości obliczone podczas propagacji w przód, takie jak wejścia, sumy ważone i wyjścia warstw. Nie przechowuje natomiast gradientów wag i biasów. Odpowiedzialność za ich wyznaczenie należy do algorytmu uczenia - w tym przypadku `BackpropagationTrainer`. Trener wykorzystuje stan zapisany podczas propagacji w przód, przeprowadza backpropagation i przekazuje obliczone gradienty do optymalizatora, który aktualizuje parametry sieci.

In [9]:
import numpy as np

from activation_functions import Sigmoid, Tanh
from neural_network import NeuralNetwork

network = NeuralNetwork(
    network_structure=[2, 4, 1],
    activation_functions=[Tanh(), Sigmoid()],
    rng=np.random.default_rng(42),
)

Lista `activation_functions` zawiera jedną aktywację dla każdej warstwy poza wejściową. YANN udostępnia aktywacje:
- `Linear`,
- `ReLU`,
- `Sigmoid`,
- `Softmax`,
- `Tanh`.

### Inicjalizacja parametrów
Wagi i biasy są inicjalizowane automatycznie:
- warstwy z `ReLU` korzystają z inicjalizacji He,
- pozostałe warstwy korzystają z inicjalizacji Xavier/Glorot,
- biasy są inicjalizowane zerami.

Wylosowane wartości wag zależą od generatora rng. Użycie tego samego ziarna pozwala odtworzyć początkowy stan sieci.

## Trening

### Podział na batche
Trening realizuje `BackpropagationTrainer`. Dane są dzielone na batche, a każdy batch jest przetwarzany wektorowo.

Dla każdego batcha trener:
1. wykonuje propagację w przód,
2. wyznacza gradient funkcji straty,
3. przeprowadza backpropagation,
4. przekazuje gradienty do optymalizatora,
5. aktualizuje wagi i biasy.

Parametr `batch_size` określa liczbę próbek przetwarzanych przed pojedynczą aktualizacją parametrów:
- `batch_size=1` - aktualizacja po każdej próbce,
- `batch_size=len(X)` - jedna aktualizacja po przetworzeniu całego zbioru,
- wartości pośrednie - trening mini-batch.

Jeśli liczba próbek nie dzieli się przez `batch_size`, ostatni batch jest po prostu mniejszy. Większy batch pozwala przetwarzać więcej danych jedną operacją macierzową, ale wymaga więcej pamięci i oznacza mniej aktualizacji parametrów w jednej epoce.
Opcja `shuffle=True` losuje kolejność próbek przed każdą epoką i przed podziałem na batche. Dla szeregów czasowych należy używać `shuffle=False`, aby zachować kolejność obserwacji.

### Kryteria zatrzymania

Jedna epoka oznacza jednokrotne przetworzenie wszystkich próbek zbioru treningowego. Liczba aktualizacji parametrów w jednej epoce wynosi `ceil(len(X) / batch_size)`. Po zakończeniu wszystkich aktualizacji w epoce YANN ponownie wykonuje propagację w przód na całym zbiorze treningowym i oblicza aktualny training loss.

Kryteria stopu przekazujemy jako listę:
```python
from stop_criteria import MaxEpochs, MaxTrainingTime, TargetLoss

stop_criteria = [
    MaxEpochs(100),
    MaxTrainingTime(30),
    TargetLoss(0.01),
]
```

Trening kończy się, gdy zostanie spełnione dowolne kryterium.

Kryteria są sprawdzane przed rozpoczęciem następnej epoki. `MaxTrainingTime` może zatem przekroczyć zadany limit mniej-więcej o czas potrzebny na dokończenie bieżącej epoki. `TargetLoss` korzysta z training loss obliczonego po zakończeniu poprzedniej epoki.

### Optymalizatory
Backpropagation wyznacza gradienty funkcji straty względem wag i biasów, ale sam nie określa, w jaki sposób parametry mają zostać zmienione. Odpowiada za to optymalizator.
Po przetworzeniu każdego batcha `BackpropagationTrainer` przekazuje optymalizatorowi:
- aktualne wartości parametrów,
- odpowiadające im gradienty,
- identyfikator parametru, np. wagi lub biasy konkretnej warstwy.

Optymalizator aktualizuje parametry bezpośrednio w sieci. Podstawowym parametrem wszystkich optymalizatorów jest `learning_rate`, który określa wielkość kroku uczenia. Zbyt duży `learning_rate` może powodować niestabilność i omijanie minimum funkcji straty. Zbyt mały może natomiast znacznie spowolnić trening.

Wbudowane optymalizatory to:.
| Optymalizator | Najważniejsze cechy mechanizmu |
|---|---|
| `SGD` | Aktualizuje parametr wyłącznie na podstawie bieżącego gradientu: `parameter -= learning_rate * gradient`. Nie przechowuje stanu ani informacji o poprzednich aktualizacjach. |
| `Momentum` | Przechowuje dla każdego parametru prędkość, która łączy poprzedni kierunek ruchu z bieżącym gradientem. Przyspiesza ruch w stabilnym kierunku i ogranicza oscylacje gradientu. Głównym parametrem jest `momentum`. |
| `NesterovMomentum` | Rozszerza momentum o korektę wykorzystującą poprzednią i nową prędkość. Pozwala wcześniej skorygować zbyt silny ruch wynikający ze zgromadzonego momentum i szybciej reagować na zmianę kierunku gradientu. |
| `RMSProp` | Przechowuje wykładniczą średnią kwadratów gradientów. Dzieli bieżący gradient przez pierwiastek tej średniej, dzięki czemu każdy element wag i biasów otrzymuje własną skalę kroku. Parametr `rho` określa długość pamięci, a `epsilon` zapewnia stabilność numeryczną. |
| `Adam` | Łączy średnią gradientów, pełniącą rolę momentum, ze średnią ich kwadratów, używaną do adaptacyjnego skalowania kroku. Stosuje korektę początkowego obciążenia obu średnich. Parametry `beta1` i `beta2` sterują długością ich pamięci. |
| `AdamW` | Wykonuje aktualizację Adama, a dodatkowo niezależnie zmniejsza wartości parametrów przez `weight_decay`. Oddzielenie decay od gradientu funkcji straty daje bezpośrednią kontrolę nad siłą tej regularyzacji. W YANN decay obejmuje zarówno wagi, jak i biasy. |

`SGD` korzysta tylko z bieżącego gradientu. `Momentum` i `NesterovMomentum` zapamiętują kierunek wcześniejszych aktualizacji. `RMSProp` dostosowuje wielkość kroku na podstawie wcześniejszych wartości gradientu. `Adam` łączy oba podejścia, natomiast `AdamW` dodaje do Adama niezależny weight decay.
Poza `SGD` wszystkie wymienione optymalizatory przechowują stan osobno dla każdego zestawu wag i biasów. Stan jest zachowywany pomiędzy batchami i epokami, a można go usunąć za pomocą `optimizer.reset_state()`.

### Funkcje straty a aktywacja wyjściowa

Po zakończeniu każdej epoki YANN oblicza wartość training loss na całym zbiorze treningowym. Ta wartość jest używana przez kryteria takie jak `TargetLoss`. W trakcie przetwarzania batcha trener wyznacza gradient funkcji straty, ale nie zapisuje osobnej wartości loss dla każdego batcha.

Aktywacja wyjściowa, funkcja straty oraz format danych Y muszą być ze sobą zgodne:
| Problem | Loss | Aktywacja wyjściowa | Format celu |
|---|---|---|---|
| Regresja, w tym szeregi czasowe | `MSE` | `Linear` | wartości rzeczywiste |
| Klasyfikacja binarna | `BinaryCrossEntropy` | `Sigmoid` | etykiety `0/1` |
| Klasyfikacja multilabel | `BinaryCrossEntropy` | `Sigmoid` | osobna etykieta `0/1` dla każdego wyjścia |
| Klasyfikacja wieloklasowa | `SparseCategoricalCrossEntropy` | `Softmax` | pojedynczy indeks klasy dla każdej próbki |

Nie jest to formalny kontrakt jednej klasy ani ograniczenie mówiące, że każdy loss może współpracować tylko z jedną aktywacją. Jest to zgodność dotycząca:
- formatu wartości docelowych,
- zakresu predykcji,
- kształtu wyjścia,
- sposobu wyznaczania gradientu.

Przykładowo `BinaryCrossEntropy` oczekuje prawdopodobieństw, dlatego naturalnie łączy się z `Sigmoid`. `Linear` nie ogranicza zakresu wyniku, dlatego pasuje do regresji. `Softmax` tworzy wektor prawdopodobieństw sumujących się do 1, co odpowiada klasyfikacji wieloklasowej.

YANN udostępnia trzy wbudowane funkcje straty:
- `MSE`,
- `BinaryCrossEntropy`,
- `SparseCategoricalCrossEntropy`.

Zwracają one średni błąd - `MSE` i `BinaryCrossEntropy `uśredniają błąd po próbkach i wyjściach, natomiast `SparseCategoricalCrossEntropy` uśrednia błąd po próbkach.

YANN korzysta z uproszczonego, stabilnego numerycznie gradientu warstwy wyjściowej. Dla pozostałych połączeń najpierw obliczany jest gradient lossu względem predykcji, a następnie jest on przekazywany przez pochodną aktywacji.

In [10]:
from losses import BinaryCrossEntropy
from optimizers import Adam
from stop_criteria import MaxEpochs
from trainers import BackpropagationOptions, BackpropagationTrainer

trainer = BackpropagationTrainer(
    loss=BinaryCrossEntropy(),
    optimizer=Adam(learning_rate=0.05),
    options=BackpropagationOptions(
        batch_size=4,
        stop_criteria=[MaxEpochs(100)],
        shuffle=False,
    ),
    rng=np.random.default_rng(43),
)

Sieć i trener powinny otrzymywać osobne generatory, ponieważ odpowiadają za różne źródła losowości:
- generator sieci odpowiada za inicjalizację wag,
- generator trenera odpowiada za tasowanie próbek.

Oddzielne generatory zapobiegają sytuacji, w której liczba losowań wykonanych podczas inicjalizacji wpływa na późniejszą kolejność próbek.

### Obserwowanie treningu
Obserwatory oddzielają mechanizm treningu od sposobu prezentowania jego postępu.

In [11]:
from observers import ConsoleObserver, HistoryObserver

trainer.add_observer(ConsoleObserver())

`ConsoleObserver` wypisuje numer epoki, aktualny loss i czas treningu. Do dyspozycji jest także `HistoryObserver`, który zapisuje wartości lossu w `history.losses`.

Po zakończeniu każdej epoki trener tworzy `EpochEndEvent`, który zawiera:
- numer ukończonej epoki,
- aktualny training loss,
- czas treningu,
- snapshot sieci.

Snapshot zawiera kopię struktury sieci, wag, biasów i nazw funkcji aktywacji z końca epoki.

Obserwatory mogą być wykorzystane do aktualizowania interfejsu użytkownika, na przykład:
- paska postępu,
- licznika epok,
- czasu treningu,
- wykresu lossu,
- wizualizacji wag lub parametrów sieci.

Callback observera jest wykonywany synchronicznie w pętli treningowej. Kosztowna obsługa zdarzenia może więc spowolnić trening. Jeżeli aplikacja ma osobny wątek UI, observer powinien zwykle umieszczać informacje w kolejce zdarzeń, a właściwa aktualizacja kontrolek powinna odbywać się w wątku interfejsu.

### Uruchomienie treningu

Rozważamy problem logiczny XOR. XOR zawiera wszystkie cztery kombinacje dwóch wejść binarnych. Jego wartość docelowa wynosi 1, gdy wejścia się różnią, i 0 w przeciwnym razie. Ponieważ problem XOR nie jest liniowo separowalny, jego rozwiązanie wymaga warstwy ukrytej.

In [12]:
# logical XOR training set.
X = np.array(
    [
        [0.0, 0.0],
        [0.0, 1.0],
        [1.0, 0.0],
        [1.0, 1.0],
    ],
    dtype=np.float32,
)
Y = np.array([0.0, 1.0, 1.0, 0.0], dtype=np.float32)

In [13]:
# training
result = network.train(trainer, X, Y)

Epoch 1: loss=0.662867; elapsed=0.00s
Epoch 2: loss=0.650813; elapsed=0.00s
Epoch 3: loss=0.637218; elapsed=0.02s
Epoch 4: loss=0.623471; elapsed=0.02s
Epoch 5: loss=0.609843; elapsed=0.02s
Epoch 6: loss=0.595766; elapsed=0.03s
Epoch 7: loss=0.580650; elapsed=0.04s
Epoch 8: loss=0.564269; elapsed=0.04s
Epoch 9: loss=0.546672; elapsed=0.04s
Epoch 10: loss=0.527940; elapsed=0.05s
Epoch 11: loss=0.508117; elapsed=0.05s
Epoch 12: loss=0.487260; elapsed=0.05s
Epoch 13: loss=0.465522; elapsed=0.05s
Epoch 14: loss=0.443167; elapsed=0.06s
Epoch 15: loss=0.420540; elapsed=0.06s
Epoch 16: loss=0.397992; elapsed=0.07s
Epoch 17: loss=0.375815; elapsed=0.07s
Epoch 18: loss=0.354211; elapsed=0.07s
Epoch 19: loss=0.333293; elapsed=0.07s
Epoch 20: loss=0.313115; elapsed=0.07s
Epoch 21: loss=0.293692; elapsed=0.08s
Epoch 22: loss=0.275014; elapsed=0.08s
Epoch 23: loss=0.257072; elapsed=0.08s
Epoch 24: loss=0.239872; elapsed=0.08s
Epoch 25: loss=0.223447; elapsed=0.08s
Epoch 26: loss=0.207836; elapsed=0

In [14]:
print(f"Epochs: {result.epochs_completed}")
print(f"Final loss: {result.final_loss:.6f}")

Epochs: 100
Final loss: 0.005161


`network.train(...)` zwraca `BackpropagationResult`, który zawiera:
- `epochs_completed` — liczbę ukończonych epok,
- `final_loss` — końcową wartość funkcji straty,
- `elapsed_seconds` — czas treningu.

Metoda `compute_outputs(X)` wykonuje wektorową propagację w przód i zwraca surowe wyjścia sieci. Dla klasyfikacji binarnej z `Sigmoid` są to prawdopodobieństwa, które można zamienić na klasy przez zastosowanie progu, np. 0.5.

### Wektoryzacja obliczeń
YANN przetwarza cały batch jednocześnie za pomocą operacji NumPy. Propagacja w przód przez warstwę gęstą ma postać:
```python
Z = X @ W + b
A = activation(Z)
```
Wymiary tablic wynoszą:
- `X: (batch_size, liczba_wejść)`,
- `W: (liczba_wejść, liczba_neuronów)`,
- `b: (liczba_neuronów,)`,
- `A: (batch_size, liczba_neuronów)`.

Bias jest dodawany do wszystkich próbek dzięki broadcastingowi NumPy. Funkcje aktywacji i funkcje straty również działają na całych tablicach, a nie osobno dla każdego neuronu lub przykładu.

Backpropagation również wykorzystuje operacje macierzowe. Dla każdej warstwy najpierw obliczany jest lokalny gradient delta, uwzględniający gradient otrzymany z następnej warstwy oraz pochodną funkcji aktywacji. Następnie wyznaczane są gradienty wag i biasów oraz gradient przekazywany do poprzedniej warstwy:
```python
dW = X.T @ delta
db = np.sum(delta, axis=0)
dX = delta @ W.T
```

`dW` i `db` służą optymalizatorowi do aktualizacji parametrów bieżącej warstwy, natomiast `dX` pozwala kontynuować propagację gradientu w kierunku wejścia sieci.

Dla pełnej ścisłości:
- `delta` oznacza gradient względem wartości przed aktywacją, czyli `dL/dZ`,
- `dW` to `dL/dW`,
- `db` to `dL/db`,
- `dX` to `dL/dX`.

Najbardziej kosztowne obliczenia są więc wykonywane przez zoptymalizowane operacje NumPy. W implementacji pozostają pętle po batchach i warstwach, ale podczas propagacji w przód i wstecz nie ma pętli po pojedynczych próbkach.

## Interpretacja wyjścia sieci

`compute_outputs(X)` zwraca surowe wyjście ostatniej warstwy. Sposób zamiany wyniku na właściwą predykcję zależy od problemu.

Dla regresji wyjście jest bezpośrednią predykcją:
```python
predictions = network.compute_outputs(X)
```
Dla klasyfikacji binarnej prawdopodobieństwo zamieniamy na klasę za pomocą wybranego progu:

```python
probabilities = network.compute_outputs(X).ravel()
predictions = (probabilities >= 0.5).astype(int)
```

Próg 0.5 nie jest częścią `Sigmoid` ani `compute_outputs()`. Jest decyzją podejmowaną podczas interpretacji wyniku i może zostać zmieniony.

W klasyfikacji multilabel próg stosujemy niezależnie do każdego wyjścia:
```python
probabilities = network.compute_outputs(X)
predictions = (probabilities >= 0.5).astype(int)
```
W klasyfikacji wieloklasowej wybieramy klasę o najwyższym prawdopodobieństwie:
```python
probabilities = network.compute_outputs(X)
predictions = np.argmax(probabilities, axis=1)
```

Dla przykładu XOR wyjście możemy zinterpretowac zatem jako:

In [15]:
# prediction
probabilities = network.compute_outputs(X).ravel()
predictions = (probabilities >= 0.5).astype(int)

print("Probabilities:", np.round(probabilities, 3))
print("Predictions:", predictions)

Probabilities: [0.006 0.997 0.994 0.005]
Predictions: [0 1 1 0]


### Funkcja straty a metryka
Loss służy do uczenia modelu i musi posiadać gradient. Metryka służy do oceny jakości modelu i nie musi być różniczkowalna.
Model regresyjny może być przykładowo trenowany za pomocą MSE, a oceniany jednocześnie przez:
- `MeanAbsoluteError`,
- `MeanSquaredError`,
- `RootMeanSquaredError`.

Dla klasyfikacji dostępne są między innymi:
- `Accuracy`,
- `BinaryAccuracy`,
- `BinaryCrossEntropyMetric`.

Loss i metryka nie muszą być tym samym.

In [16]:
from metrics import (
    BinaryAccuracy,
    BinaryCrossEntropy as BinaryCrossEntropyMetric,
)

accuracy_metric = BinaryAccuracy()
cross_entropy_metric = BinaryCrossEntropyMetric()

accuracy = accuracy_metric.compute(Y, probabilities)
cross_entropy = cross_entropy_metric.compute(Y, probabilities)

print(f"Binary accuracy: {accuracy:.3f}")
print(f"Binary cross-entropy: {cross_entropy:.6f}")
print(f"Final training loss: {result.final_loss:.6f}")

Binary accuracy: 1.000
Binary cross-entropy: 0.005161
Final training loss: 0.005161


Metryki otrzymują wartości docelowe oraz surowe predykcje modelu. `BinaryAccuracy` samodzielnie stosuje próg 0.5, dlatego przekazujemy do niego prawdopodobieństwa, a nie wcześniej wyznaczone klasy. `BinaryCrossEntropy` jako metryka oblicza tę samą wielkość matematyczną co loss o tej nazwie, ale nie uczestniczy w backpropagation — służy wyłącznie do oceny wyniku.

W tym przykładzie metryki są liczone na danych treningowych, dlatego pokazują stopień dopasowania do czterech przykładów XOR, a nie zdolność modelu do generalizacji na nowych danych.

# Rozszerzanie YANN
YANN można rozbudowywać bez modyfikowania głównego mechanizmu sieci i trenera. Własne komponenty tworzymy przez implementację kontraktów odpowiednich klas bazowych, między innymi:
- `Activation`,
- `Loss`,
- `Optimizer`,
- `StopCriterion`,
- `BackpropagationObserver`,
- `Metric`,
- `DataTransformer`,
- `Splitter`.

Przykładowo własna funkcja straty, taka jak Huber Loss, powinna dziedziczyć po `Loss` i implementować obliczenie wartości oraz gradientu dla batcha.
Dzięki publicznym kontraktom można dodawać własne aktywacje, funkcje straty, optymalizatory, kryteria zatrzymania, obserwatory, metryki i strategie podziału bez kopiowania implementacji backpropagation.